# Fase 0 · Explorando los datos de SPHEREx

Objetivo: dadas unas coordenadas, (1) preguntar a IRSA qué observaciones las cubren, (2) bajar un *cutout* y (3) entender qué hay dentro de cada archivo FITS.

Resumen de lo que vas a encontrar:

- Cada observación es un archivo **MEF** (multi-extension FITS) de ~70 MB por detector y exposición. El cutout devuelve las mismas extensiones pero recortadas.
- Extensiones: `IMAGE`, `FLAGS`, `VARIANCE`, `ZODI`, `PSF`, `WCS-WAVE`.
- Cada **píxel** tiene su propia longitud de onda (el filtro es de variación lineal), así que una imagen NO es una sola banda: la longitud de onda cambia a lo largo de la imagen.
- Los 6 detectores (`SPHEREx-D1` … `SPHEREx-D6`) cubren distintos rangos de longitud de onda.

Fuentes: [Data Access](https://caltech-ipac.github.io/spherex-archive-documentation/spherex-data-access/), [Data Products](https://caltech-ipac.github.io/spherex-archive-documentation/spherex-data-products/) y el tutorial [Spectral Image cutouts](https://caltech-ipac.github.io/irsa-tutorials/spherex-cutouts/).

In [ ]:
# !pip install numpy astropy astroquery matplotlib
import time
import logging
import concurrent.futures

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.time import Time
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval
from astroquery.ipac.irsa import Irsa
from astropy.utils.data import conf

logging.getLogger("astropy").setLevel(logging.ERROR)  # evita avisos repetidos de WCS alternos
conf.remote_timeout = 120                             # los archivos SPHEREx pueden tardar en leerse

## 1. Objetivo y tamaño del recorte

WISE 0855 viene con las coordenadas del enunciado. Los otros tres se resuelven por nombre con SIMBAD (necesita internet).

Dato útil para más adelante: WISE 0855 tiene un movimiento propio muy grande (del orden de 8″ por año), mientras que un píxel de SPHEREx mide ~6.2″. Es decir, en un año se mueve poco más de un píxel: el efecto es sutil y por eso hace falta alinear bien las imágenes.

In [ ]:
# Respaldo por si SIMBAD/Sesame no responde (timeouts de red).
# eps Ind A y RW Aur: coordenadas J2000/ICRS tomadas de Wikipedia.
# eps Ind B y R Dra: NO las tengo verificadas. Si Sesame falla, busca "eps Ind B" y "R Dra" en
# https://simbad.cds.unistra.fr/simbad/ y descomenta/pega aquí sus coordenadas.
FALLBACK = {
    "eps Ind A": SkyCoord("22h03m21.65363s", "-56d47m09.5228s", frame="icrs"),
    "RW Aur": SkyCoord("05h07m49.4562s", "+30d24m04.774s", frame="icrs"),
    # "eps Ind B": SkyCoord("hh mm ss.s", "+dd mm ss.s", frame="icrs"),
    # "R Dra": SkyCoord("hh mm ss.s", "+dd mm ss.s", frame="icrs"),
}


def resolve(name, retries=1):
    for _ in range(retries + 1):
        try:
            return SkyCoord.from_name(name)
        except Exception as e:
            err = e
            time.sleep(2)
    print(f"[{name}] Sesame no respondió ({type(err).__name__}); uso respaldo si existe.")
    return FALLBACK.get(name)


TARGETS = {
    "WISE 0855": SkyCoord("08h55m10.831s", "-07d14m42.53s", frame="icrs"),
}
for name in ["eps Ind A", "eps Ind B", "RW Aur", "R Dra"]:
    c = resolve(name)
    if c is None:
        print(f"[{name}] sin coordenadas: pégalas a mano en FALLBACK.")
    else:
        TARGETS[name] = c

for k, v in TARGETS.items():
    print(f"{k:10s} RA={v.ra.deg:.5f}  Dec={v.dec.deg:+.5f}")

TARGET = "WISE 0855"       # cambia aquí para probar los otros
coord = TARGETS[TARGET]
SIZE = 0.1 * u.deg         # lado del recorte

## 2. ¿Qué observaciones cubren este punto? (TAP)

Se usa el servicio TAP de IRSA. La tabla `spherex.plane` tiene el polígono de cada observación (`poly`), su detector (`energy_bandpassname`) y su fecha (`time_bounds_lower`); `spherex.artifact` tiene la URL del archivo. A la URL se le añade `?center=RA,Dec&size=...` y IRSA devuelve el cutout.

La consulta puede tardar unos 15 segundos.

In [ ]:
def query_observations(coord, size=SIZE, bandpass=None):
    cond = f"AND p.energy_bandpassname = '{bandpass}'" if bandpass else ""
    query = f"""
    SELECT
        'https://irsa.ipac.caltech.edu/' || a.uri || '?center={coord.ra.deg},{coord.dec.deg}d&size={size.value}' AS uri,
        p.time_bounds_lower,
        p.energy_bandpassname
    FROM spherex.artifact a
    JOIN spherex.plane p ON a.planeid = p.planeid
    WHERE 1 = CONTAINS(POINT('ICRS', {coord.ra.deg}, {coord.dec.deg}), p.poly)
    {cond}
    ORDER BY p.time_bounds_lower
    """
    return Irsa.query_tap(query).to_table()


t0 = time.time()
obs = query_observations(coord)
print(f"{len(obs)} observaciones en {time.time() - t0:.1f} s")
obs[:5]

In [ ]:
# Resumen: cuántas observaciones por detector y qué rango de fechas
# (time_bounds_lower está en MJD; los años impresos deberían ser 2025-2026)
obs["date"] = Time(np.array(obs["time_bounds_lower"], dtype=float), format="mjd").iso
for det in np.unique(obs["energy_bandpassname"]):
    sel = obs[obs["energy_bandpassname"] == det]
    print(f"{det}: {len(sel):3d} obs   {sel['date'][0][:10]} -> {sel['date'][-1][:10]}")

n_dates = len(np.unique([d[:10] for d in obs["date"]]))
print(f"\nFechas distintas: {n_dates}")

**Qué mirar aquí:** si hay muy pocas fechas distintas, el “video” será corto. Guarda este número para cada uno de los 4 objetivos; define qué tan bien se verá el movimiento.

## 3. Bajar un cutout y ver su estructura

Se toma la primera observación del detector D2 (cualquier otra sirve).

In [ ]:
row = obs[obs["energy_bandpassname"] == "SPHEREx-D2"][0]
url = row["uri"]
print(url)

hdul = fits.open(url)   # descarga el recorte (queda en caché de astropy)
hdul.info()

In [ ]:
# Header primario: metadatos generales de la observación (sin datos de imagen)
for k, v in list(hdul[0].header.items())[:45]:
    print(f"{k:10s} = {v}")

In [ ]:
# Header de IMAGE: unidades y WCS espacial
hdr = hdul["IMAGE"].header
for key in ["EXTNAME", "NAXIS1", "NAXIS2", "BUNIT", "CTYPE1", "CTYPE2",
            "CRVAL1", "CRVAL2", "CRPIX1", "CRPIX2", "CD1_1", "CD1_2", "CD2_1", "CD2_2"]:
    print(f"{key:8s} = {hdr.get(key)}")
print("\nTamaño de cada extensión:")
for h in hdul:
    print(f"  {h.name:9s} {h.data.shape if h.data is not None else None}")

**Anota:** la unidad (`BUNIT`, debería ser MJy/sr), el tamaño en píxeles del recorte y la extensión `PSF`, que NO es una imagen del cielo (su WCS es de ángulos del detector, no RA/Dec).

## 4. Las capas: IMAGE, VARIANCE, ZODI y FLAGS

Se marca con una cruz la posición del objetivo usando el WCS del propio recorte.

In [ ]:
spatial_wcs = WCS(hdul["IMAGE"].header)
x0, y0 = spatial_wcs.world_to_pixel(coord)
print(f"Objetivo en el píxel (x={x0:.1f}, y={y0:.1f})")


def show(ax, data, title, log=False):
    d = np.array(data, dtype=float)
    ok = np.isfinite(d)
    vmin, vmax = ZScaleInterval().get_limits(d[ok]) if ok.any() else (0, 1)
    im = ax.imshow(d, origin="lower", vmin=vmin, vmax=vmax, cmap="viridis")
    ax.plot(x0, y0, "r+", ms=14)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)


fig, axs = plt.subplots(2, 2, figsize=(11, 10))
show(axs[0, 0], hdul["IMAGE"].data, "IMAGE (MJy/sr)")
show(axs[0, 1], hdul["VARIANCE"].data, "VARIANCE ((MJy/sr)^2)")
show(axs[1, 0], hdul["ZODI"].data, "ZODI (luz zodiacal modelada)")
show(axs[1, 1], hdul["FLAGS"].data, "FLAGS (bitmap)")
plt.tight_layout()
plt.show()

**Nota sobre `ZODI`:** es un modelo del fondo de luz zodiacal que viene como referencia y **no está restado** de `IMAGE`. Si quieres ver el objeto sin fondo, esa es una decisión de procesamiento tuya (restar el modelo, o restar una mediana, etc.).

In [ ]:
# FLAGS es un bitmap: cada bit significa algo (saturación, rayo cósmico, fuente enmascarada, etc.).
# Aquí solo vemos qué bits aparecen y en qué fracción de píxeles.
flags = np.asarray(hdul["FLAGS"].data).astype(np.int64)
print("bits activos:")
for b in range(32):
    frac = np.mean((flags & (1 << b)) > 0)
    if frac > 0:
        print(f"  bit {b:2d}: {100 * frac:6.2f}% de los píxeles")

**Pendiente (importante):** el significado exacto de cada bit está en el [Explanatory Supplement](https://irsa.ipac.caltech.edu/data/SPHEREx/docs/SPHEREx_Expsupp_QR.pdf). Cópialo a una tabla en tu código: sirve para la capa de “máscara de contaminación” de la web.

## 5. Longitud de onda por píxel

Esta es la parte más rara de SPHEREx: el filtro varía a lo largo del detector, así que cada píxel tiene su propia longitud de onda y ancho de banda. El WCS espectral está en `WCS-WAVE` y se lee con `key="W"`.

Aviso de la documentación: `WCS-WAVE` es una aproximación pensada para visualización (~1 nm); para ciencia se recomienda el producto aparte *Spectral WCS* (extensiones `CWAVE` y `CBAND`, un archivo de ~32 MB por detector).

In [ ]:
ny, nx = hdul["IMAGE"].data.shape
spec_wcs = WCS(hdul["IMAGE"].header, fobj=hdul, key="W")
spec_wcs.sip = None

yy, xx = np.mgrid[0:ny, 0:nx]
wave, bandwidth = spec_wcs.pixel_to_world(xx, yy)
wave_um = wave.to(u.micrometer).value
bw_um = bandwidth.to(u.micrometer).value

wl_target = wave_um[int(round(y0)), int(round(x0))]
print(f"Longitud de onda en el objetivo: {wl_target:.4f} um")
print(f"Rango de longitud de onda en el recorte: {wave_um.min():.4f} - {wave_um.max():.4f} um")

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
im = axs[0].imshow(wave_um, origin="lower", cmap="plasma")
axs[0].plot(x0, y0, "w+", ms=14)
axs[0].set_title("Longitud de onda por píxel (um)")
plt.colorbar(im, ax=axs[0], fraction=0.046)
im = axs[1].imshow(bw_um, origin="lower", cmap="magma")
axs[1].set_title("Ancho de banda por píxel (um)")
plt.colorbar(im, ax=axs[1], fraction=0.046)
plt.tight_layout()
plt.show()

**Qué significa para el proyecto:** en cada exposición, el objetivo cae en UNA longitud de onda concreta. Para tener las 102 bandas hay que juntar exposiciones distintas. Ahí está la idea del “espectro instantáneo”: es el flujo del objetivo en cada exposición, graficado contra la longitud de onda de ese píxel.

## 6. Un vistazo a los 6 detectores

Para cada detector se toma su primera observación y se calcula qué longitud de onda ve el objetivo.

In [ ]:
def inspect_obs(row, coord):
    """Devuelve fecha, longitud de onda y flujo (mediana 3x3) en el objetivo para un cutout."""
    with fits.open(row["uri"]) as h:
        wcs_s = WCS(h["IMAGE"].header)
        x, y = wcs_s.world_to_pixel(coord)
        wcs_w = WCS(h["IMAGE"].header, fobj=h, key="W")
        wcs_w.sip = None
        wl, _ = wcs_w.pixel_to_world(x, y)
        img = np.array(h["IMAGE"].data, dtype=float)
        xi, yi = int(round(float(x))), int(round(float(y)))
        patch = img[max(yi - 1, 0):yi + 2, max(xi - 1, 0):xi + 2]
        return {
            "det": row["energy_bandpassname"],
            "date": row["date"][:10],
            "wave_um": float(wl.to(u.micrometer).value),
            "flux_MJy_sr": float(np.nanmedian(patch)),
        }


first_per_det = [obs[obs["energy_bandpassname"] == d][0] for d in np.unique(obs["energy_bandpassname"])]
for r in first_per_det:
    try:
        print(inspect_obs(r, coord))
    except Exception as e:
        print(r["energy_bandpassname"], "falló:", e)

## 7. Varias épocas: el germen del espectro y del video

Se procesan las primeras observaciones en paralelo. IRSA recomienda no pasarse de ~10 hilos. Si alguna descarga falla, es normal (errores transitorios): la función la salta.

In [ ]:
def safe_inspect(row):
    try:
        return inspect_obs(row, coord)
    except Exception:
        return None


subset = obs[:24]
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    results = [r for r in ex.map(safe_inspect, subset) if r is not None]

print(f"{len(results)} de {len(subset)} cutouts leídos")
results.sort(key=lambda r: r["wave_um"])

waves = [r["wave_um"] for r in results]
flux = [r["flux_MJy_sr"] for r in results]
plt.figure(figsize=(8, 4))
plt.plot(waves, flux, "o-")
plt.xlabel("Longitud de onda (um)")
plt.ylabel("Flujo (MJy/sr, mediana 3x3)")
plt.title(f"Espectro crudo de {TARGET} (primeras {len(results)} exposiciones)")
plt.show()

## Qué anotar al terminar la Fase 0

Rellena esto para cada uno de los 4 objetivos:

| Objetivo | Nº de observaciones | Nº de fechas distintas | Detectores que lo cubren | Rango de fechas |
|---|---|---|---|---|
| WISE 0855 | | | | |
| eps Ind B | | | | |
| RW Aur | | | | |
| R Dra | | | | |

Preguntas abiertas para la Fase 1:

1. ¿Cuántas fechas distintas hay? (define qué tan largo y útil será el video)
2. ¿El flujo del objetivo se ve por encima del ruido en `VARIANCE`?
3. ¿Qué bits de `FLAGS` caen sobre el objetivo?
4. ¿Cómo comparas dos imágenes de distinta longitud de onda? (si el brillo cambia con la banda, restarlas no basta)
5. ¿Qué versión de procesamiento tiene cada archivo (`VERSION` en el header primario)? Lo necesitas para la actualización a la versión final.